# MuQ: benchmark iPhone INT8 dan palette6

Membuat proyek Xcode terpisah dari SongMap. Memakai input conv/rotary nyata dari ekspor MuQ FP32: prefix 750 frame dan window panjang 3000 frame. Prefix memakai konteks input panjang; bukan hasil encoding audio 30 detik baru.

Paket setiap kandidat dibuat statis untuk tiap fixture, diperiksa paritas CPU di Mac, lalu dikompilasi. Bobot digandakan untuk isolasi benchmark; ukuran bundle uji bukan ukuran aplikasi produksi. Sediakan beberapa GiB disk. Gunakan kernel `.venv-songformer311` di root Section-Annotations. Kompilasi memerlukan Xcode di Mac.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, subprocess, sys
assert Path('finetuning/config.json').is_file()
files=sorted(Path('outputs.local').glob('muq-compression-*/comparison.json'),key=lambda p:p.stat().st_mtime,reverse=True)
assert files, 'comparison.json kompresi MuQ belum ditemukan'
comparison=files[0]
output=Path('outputs.local')/('muq-iphone-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
print('Python:',sys.executable)
print('Comparison:',comparison)
print('Output:',output)


In [ ]:
log=output.with_suffix('.prepare.log')
log.parent.mkdir(parents=True,exist_ok=True)
command=[sys.executable,'-u','-m','deployment.prepare_muq_iphone',
         '--comparison',str(comparison),'--output-dir',str(output)]
with log.open('w',encoding='utf-8') as handle:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='');handle.write(line);handle.flush()
        status=process.wait()
    except KeyboardInterrupt:
        process.terminate();process.wait();raise
if status:
    raise RuntimeError(f'Persiapan gagal ({status}); lihat {log.resolve()}')
subprocess.run(['open',str(output/'Benchmark.xcodeproj')],check=True)


## Di Xcode dan iPhone

1. Pilih Signing Team, lalu target iPhone fisik. Jalankan proyek benchmark.
2. Mulai dengan `cpuOnly`, kandidat `palette6`, fixture `short`, kemudian `long`.
3. Ulangi untuk `int8`. Tutup dan buka aplikasi lagi antar kandidat untuk mengurangi pengaruh memori yang tertahan.
4. Ulangi keempat kombinasi dengan compute units `all`. Gunakan kondisi suhu normal dan Low Power Mode mati.
5. Tekan **Share JSON report** setelah tiap pengujian dan kirim berkasnya. Total delapan laporan untuk dua kandidat × dua fixture × dua mode.

Laporan mencatat waktu load, prediksi pertama, prediksi berulang, NaN/Inf, error terhadap referensi FP32, kondisi perangkat, dan snapshot memori proses. Snapshot tersebut bukan peak memory. Jika aplikasi berhenti/assert, kirim log Xcode beserta kandidat, fixture, dan mode yang sedang diuji. Benchmark ini hanya MuQ L10; frontend, MusicFM, head, dan keseluruhan lagu belum diukur.